# ExperimentRobustness

This notebook pits the optical knots against various experimental conditions. Partially copied from the PhaseAnalyzer notebook

In [ ]:
# Functions to import

import numpy as np 
import scipy as sp
import pygad
import yaml 
from pathlib import Path

from scipy.fft import fft2, fftfreq, ifft2, fftshift, ifftshift
from scipy import ndimage
from optical_functions import TotInt, LG, propFF, propTF, cart2pol, oamModes, output_chan, setKnotType, output_chan_symmetric, output_chan_triangle, output_chan_circle, norm_field, build_fresnel_lens_kernels, propagate_fresnel_lens_train, propagate_legacy_fft, complex_field_fidelity, intensity_fidelity
from sorter_configuration import parse_optical_train_config
#from run_ga import compute_sorting_performance

import matplotlib.pyplot as plt 

import os

# Physical Constants

nm = 1e-9
um = 1e-6
mm = 1e-3
cm = 1e-2

Load optimized solution

In [ ]:
import pickle

index = 3

#experiment_name = "Knot Sorting Bread"
#experiment_name = "Knot Sorting New FF"
#experiment_name = "Knot Sorting Three Knot Fun"
#experiment_name = "Knot Sorting New FF Smaller Alpha"  # use None for configs/ga0.yaml, ga1.yaml, or ga2.yaml
#experiment_name = "Knot Sorting New FF Large Alpha"
#experiment_name = "tref_cinque_fresnel_1plane"

experiment_name = None

config_path = Path('configs') / (f'ga{index}.yaml' if experiment_name is None else f'{experiment_name}/ga{index}.yaml')
with config_path.open('r', encoding='utf-8') as stream:
    cnfg = yaml.safe_load(stream)

cnfg.setdefault('circle_radius', 1.5)
cnfg.setdefault('alpha', 0.0)

N = cnfg['dim']
num_of_output_chans = cnfg['num_output_chans']
output_chan_width = cnfg['output_chan_width'] * mm # in mm 

num_phase_maps_near = cnfg.get('num_phase_maps_near', 0)
num_phase_maps_far = cnfg.get('num_phase_maps_far', 0)

num_of_phase_maps = cnfg.get('num_phase_planes', num_phase_maps_near + num_phase_maps_far)
optical_train = parse_optical_train_config(cnfg, num_of_phase_maps)
instance_name = cnfg['ga_instance'] # directory name of best phases

# Print the instance name (for reference)

print(instance_name)

# Some parameters specifying the LG modes

LG_modes = cnfg['LG_modes']
w0 = cnfg['w0'] * mm # in mm!!

isKnot = cnfg['isKnot']
knotType = cnfg['knotType']
shapeParams = cnfg['shapeParams']
fourier_lens = cnfg.get('fourier_length', 10.0)*cm # legacy propagation only
GFilterStrength=cnfg['gauss_filter_sigma']
channel_seperation = cnfg['channel_sep']
circle_radius = cnfg['circle_radius'] # circle radius is in mm
alpha = cnfg['alpha']

# Define the coordinate space 

la = cnfg.get('wavelength_nm', 780.0)*nm
k=(2*np.pi)/la  # [m^-1] wavenumber    
N = cnfg['dim'] # [Number of points per dimension]
maxx = cnfg.get('pixel_pitch_um', 20.0)*um*N  # Full numerical-window length (m)

# Propagation Distance 
prop_dist = 0

# Let's apply a rotation
rot_phi = eval(cnfg['rot_angle'])

# Space definition 
dx = maxx/N
dy = maxx/N 

#okay let's just say h here is dx or dy for now WLOG (WITH ... loss of generality)

h = dx
X = dx*(np.arange(N) - N //2)
Y = dy*(np.arange(N) - N //2)

# Apply rotation operator on coords 

xx,yy=np.meshgrid(X ,Y)

r, phi= cart2pol(xx, yy)

# What experiment are we going to design

simulateLens = cnfg.get('simulateLens', False)
multiPhase = cnfg.get('multiPhase', False)
multiPhaseLens = cnfg.get('multiPhaseLens', False)

z_o = cnfg.get('z_o', 30.0)*cm
fourier_lens = cnfg.get('fourier_length', 10.0)*cm

''' 
Create the OAM beams that we need to sort 
'''
# Now create a list containing 'oamMode' objects 

list_of_OAMs = []

output_chans = output_chan_circle(X, Y, output_chan_width, maxx, num_of_output_chans, circle_radius=circle_radius, coordinate_mode=optical_train.output_coordinate_mode)

if(isKnot):
    for ii in range(len(knotType)):
        field = setKnotType(r, phi, w0, knotType[ii], shapeParams[ii])
        prop_field = field
        list_of_OAMs.append(oamModes(prop_field, output_chans[ii]))
else:
    for ii in range(len(LG_modes)):
        ell, p = LG_modes[ii][0], LG_modes[ii][1]
        field = LG(r, phi, ell, p, w0, h, 0, k)
        prop_field = propTF(field, maxx, la, prop_dist)
        list_of_OAMs.append(oamModes(prop_field, output_chans[ii]))


# Load up phase screens

with open(f"best_phases/{instance_name}.pkl", 'rb') as file:
     phase_out = pickle.load(file)

geometry_path = Path('best_phases') / f'{instance_name}_geometry.yaml'
analysis_padding_factor = optical_train.padding_factor
if optical_train.model == 'fresnel_lens_train' and geometry_path.exists():
    with geometry_path.open('r', encoding='utf-8') as stream:
        saved_geometry = yaml.safe_load(stream)
    analysis_padding_factor = saved_geometry.get('padding_factor', analysis_padding_factor)
    sorter_stages = [
        {'z_to_lens': stage['z_to_lens_cm']*cm,
         'focal_length': stage['focal_length_cm']*cm,
         'z_after_lens': stage['z_after_lens_cm']*cm}
        for stage in saved_geometry['stages']
    ]
else:
    initial_geometry = optical_train.initial_normalized_geometry if optical_train.num_geometry_genes else None
    sorter_stages = optical_train.decode_geometry(initial_geometry)
    if optical_train.model == 'fresnel_lens_train':
        print(f'Warning: {geometry_path} was not found; using the YAML initial geometry, not candidate-optimized geometry.')


phase_maps = np.empty((num_of_phase_maps, N, N), dtype=np.complex128)

# Compute phase screens

for ii in range(num_of_phase_maps):
    phase_maps[ii] = np.exp(1j * phase_out[ii])

analysis_fresnel_kernels = None
if optical_train.model == 'fresnel_lens_train':
    analysis_fresnel_kernels = build_fresnel_lens_kernels(
        (N, N), maxx, la, sorter_stages, r,
        lens_radius=optical_train.lens_aperture_radius,
        padding_factor=analysis_padding_factor,
    )

def propagate_analysis_field(field):
    if optical_train.model != 'fresnel_lens_train':
        raise RuntimeError('propagate_analysis_field is for the physical Fresnel train.')
    return propagate_fresnel_lens_train(
        field, phase_maps, maxx, la, sorter_stages, r,
        lens_radius=optical_train.lens_aperture_radius,
        kernels=analysis_fresnel_kernels,
        padding_factor=analysis_padding_factor,


# Test 1:  Effects of z- propagation on sorter performance

What happens when we propagate the input knotted field longitudinally over a range z? 

# Test 2: Effects of Chirality

What happens when we sort as input a mirror image? This can be realized potentially by flipping the sign of the OAM components of the knot

# Test 3: a- and b- parameter sweep

What happens when we sweep through the a- and b- parameters of the input knotted beam? 

In [ ]:
def mold_knots(a_offset, b_offset):

    '''
    generates aberrated knots 

    m_n -- tuple indicating the Zernike mode indices (m, n)
    gamma -- strength of the Zernike modes
    aperature -- aperature size of the aberration
    '''

    list_of_OAMs = []

    if(isKnot):
        for ii in range(len(knotType)):
            shapeParams_a, shapeParams_b, shapeParams_s = shapeParams[ii]
            # Introduce the offset for a, b, and s parameters (when applicable)
            shapeParams_a += a_offset
            shapeParams_b += b_offset
            # Put everything back together
            shapes = [shapeParams_a, shapeParams_b, shapeParams_s]
            field = setKnotType(r, phi, w0, knotType[ii], shapes)
            list_of_OAMs.append(oamModes(field, output_chans[ii]))
    else:
        for ii in range(len(LG_modes)):
            field = LG(r, phi, LG_modes[ii][0], LG_modes[ii][1], w0,h,0,k)
            list_of_OAMs.append(oamModes(LG(r, phi, LG_modes[ii][0], LG_modes[ii][1], w0,h,0,k), output_chans[ii]))
    
    return list_of_OAMs


# Generate the offset in the a- and b- parameterization

a_offset = np.linspace(-1.0, 1.0, 100)
b_offset = np.linspace(-1.0, 1.0, 100)


